# validation-no-grad — worked example 2: Accuracy accumulator under no_grad

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `validation-no-grad`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A validation epoch accumulates correct-count and total-count across batches inside one `no_grad` block, then divides for accuracy. Doing the whole loop under `no_grad` keeps every intermediate graph-free, which is essential when iterating many eval batches.

## Worked solution

We compute classification accuracy over several batches without building any graph.

1. We enter `with t.no_grad():` and initialize integer `correct` and `total` accumulators.
2. For each `(xb, yb)` batch, we forward the model, take `argmax(dim=1)` as the prediction, and add the number of matches to `correct` and the batch size to `total`.
3. After the loop we return `correct / total` as the accuracy.

Because the loop runs under `no_grad`, none of the logits track gradients. We print the accumulated counts and the final accuracy.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(1)
model = nn.Linear(5, 3)
batches = [(t.randn(4, 5), t.randint(0, 3, (4,))) for _ in range(3)]

def eval_accuracy(model, batches):
    correct = 0
    total = 0
    with t.no_grad():
        for xb, yb in batches:
            preds = model(xb).argmax(dim=1)
            correct += int((preds == yb).sum())
            total += yb.numel()
    return correct / total

acc = eval_accuracy(model, batches)
print('accuracy:', round(acc, 3))
print('in range:', 0.0 <= acc <= 1.0)